In [ ]:
# Save Model and Export Outputs
import joblib

# Save model
joblib.dump(gs.best_estimator_, '../models/employee_perf_model.pkl')
print("Model saved to ../models/employee_perf_model.pkl")

# Save processed data
df.to_csv('../outputs/cleaned_data.csv', index=False)
print("Cleaned data saved.")

print("Notebook execution completed. Check images/ for plots.")

In [ ]:
# Visualize Insights and Feature Importance
from sklearn.inspection import permutation_importance

# Permutation importance
r = permutation_importance(gs.best_estimator_, X_test, y_test, n_repeats=5, scoring='f1_macro', random_state=42)
imp = pd.Series(r.importances_mean, index=gs.best_estimator_.named_steps['pre'].get_feature_names_out())
top_imp = imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10,8))
top_imp.plot(kind='barh')
plt.title('Top 15 Feature Importances (Permutation)')
plt.savefig('../images/feature_importance.png')
plt.show()

In [ ]:
# Generate Predictions and Recommendations
# Example prediction
new_employee = pd.DataFrame({
    'age': [30],
    'gender': ['Male'],
    'education': ['Master'],
    'experience_years': [5],
    'department': ['Engineering'],
    'job_level': ['Mid'],
    'manager_tenure': [3],
    'projects_count': [10],
    'avg_task_delay_days': [2.0],
    'on_time_delivery_rate': [0.85],
    'bug_count': [5],
    'code_review_score': [4.0],
    'qa_defect_density': [0.1],
    'story_points_completed': [200],
    'billable_hours_ratio': [0.9],
    'training_hours': [20.0],
    'certifications_count': [2],
    'internal_hackathons_participated': [1],
    'sick_days': [2],
    'unplanned_absences': [1],
    'avg_login_hours': [8.5],
    'peer_feedback_score': [4.2],
    'manager_score': [4.5],
    'kudos_count': [5],
    'promotions_in_2y': [1],
    'salary_percentile_band': ['Medium']
})

pred = gs.best_estimator_.predict(new_employee)
print("Predicted Performance Band:", pred[0])

# Simple recommendations based on features
if pred[0] == 'Low':
    print("Recommendations: Increase training hours, improve on-time delivery, reduce bugs.")
elif pred[0] == 'Medium':
    print("Recommendations: Focus on certifications, participate in hackathons.")
else:
    print("Recommendations: Maintain performance, consider leadership roles.")

In [ ]:
# Evaluate Model Metrics
from sklearn.metrics import classification_report, confusion_matrix

y_pred = gs.best_estimator_.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['Low', 'Medium', 'High'])
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.title('Confusion Matrix')
plt.savefig('../images/confusion_matrix.png')
plt.show()

In [ ]:
# Train Employee Performance Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Pipeline with classifier
pipe = Pipeline([
    ('pre', pre),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# Grid search
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [10, 20, None],
    'clf__min_samples_leaf': [1, 2, 4]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(pipe, param_grid, scoring='f1_macro', cv=cv, n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print("Best CV F1_macro:", gs.best_score_)
print("Best params:", gs.best_params_)

In [ ]:
# Build Preprocessing Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(include='number').columns

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

pre = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

print("Preprocessing pipeline created.")

In [ ]:
# Feature Engineering
# Split for EDA
X = df.drop(columns=['perf_band_next', 'employee_id'])
y = df['perf_band_next']

# Encode target for MI
y_encoded = y.map({'Low': 0, 'Medium': 1, 'High': 2})

# Mutual Information
num_cols = X.select_dtypes(include='number').columns
mi_scores = mutual_info_classif(X[num_cols].fillna(0), y_encoded)
mi_df = pd.Series(mi_scores, index=num_cols).sort_values(ascending=False)
print("Top 10 features by Mutual Information:")
print(mi_df.head(10))

# Plot MI
plt.figure(figsize=(10,6))
mi_df.head(10).plot(kind='barh')
plt.title('Top 10 Features by Mutual Information')
plt.savefig('../images/mutual_info.png')
plt.show()

# Categorical encoding preview (OHE would be done in pipeline)
print("Categorical columns:", X.select_dtypes(include='object').columns.tolist())

In [ ]:
# Exploratory Data Analysis
# Class balance
plt.figure(figsize=(8,6))
sns.countplot(data=df, x='perf_band_next', order=['Low', 'Medium', 'High'])
plt.title('Performance Band Distribution')
plt.savefig('../images/class_balance.png')
plt.show()

# Numeric feature distributions
num_cols = df.select_dtypes(include='number').columns
df[num_cols].hist(figsize=(20,15), bins=30)
plt.tight_layout()
plt.savefig('../images/feature_histograms.png')
plt.show()

# Correlation heatmap
plt.figure(figsize=(12,10))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title('Feature Correlations')
plt.savefig('../images/correlation_heatmap.png')
plt.show()

# Target vs key features
key_features = ['on_time_delivery_rate', 'manager_score', 'training_hours', 'experience_years']
fig, axes = plt.subplots(2, 2, figsize=(12,10))
for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x='perf_band_next', y=feat, ax=axes[i//2, i%2], order=['Low', 'Medium', 'High'])
plt.tight_layout()
plt.savefig('../images/target_vs_features.png')
plt.show()

In [ ]:
# Data Cleaning and Missing Value Handling
# For this synthetic data, assume no missing values, but in real data:
# df.fillna(method='median' for numeric, 'mode' for categorical)

# Ensure categorical consistency
categorical_cols = ['gender', 'education', 'department', 'job_level', 'salary_percentile_band']
for col in categorical_cols:
    print(f"{col} unique values: {df[col].unique()}")

# No cleaning needed for synthetic data
print("Data cleaning completed.")

In [ ]:
# Load Dataset and Run Sanity Checks
df = pd.read_csv('../data/employee_features.csv')
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())
print("\nUnique employee IDs:", df['employee_id'].nunique() == len(df))

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Basic range checks
rules = {
    'experience_years': (0, 45),
    'on_time_delivery_rate': (0, 1),
    'manager_score': (0, 5),
    'peer_feedback_score': (0, 5),
    'avg_task_delay_days': (-10, 60),
}
for col, (lo, hi) in rules.items():
    bad = df[(df[col] < lo) | (df[col] > hi)]
    print(f"{col} out_of_range: {len(bad)}")

In [ ]:
# Create Synthetic Employee Dataset
np.random.seed(42)
n = 1000

data = {
    'employee_id': range(1, n+1),
    'age': np.random.randint(22, 60, n),
    'gender': np.random.choice(['Male', 'Female'], n),
    'education': np.random.choice(['Bachelor', 'Master', 'PhD'], n, p=[0.5, 0.4, 0.1]),
    'experience_years': np.random.randint(0, 40, n),
    'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Finance', 'Marketing'], n),
    'job_level': np.random.choice(['Junior', 'Mid', 'Senior'], n),
    'manager_tenure': np.random.randint(1, 20, n),
    'projects_count': np.random.randint(1, 20, n),
    'avg_task_delay_days': np.random.uniform(-5, 30, n),
    'on_time_delivery_rate': np.random.uniform(0.5, 1.0, n),
    'bug_count': np.random.randint(0, 50, n),
    'code_review_score': np.random.uniform(1, 5, n),
    'qa_defect_density': np.random.uniform(0, 0.5, n),
    'story_points_completed': np.random.randint(50, 500, n),
    'billable_hours_ratio': np.random.uniform(0.6, 1.0, n),
    'training_hours': np.random.uniform(0, 100, n),
    'certifications_count': np.random.randint(0, 10, n),
    'internal_hackathons_participated': np.random.randint(0, 5, n),
    'sick_days': np.random.randint(0, 20, n),
    'unplanned_absences': np.random.randint(0, 10, n),
    'avg_login_hours': np.random.uniform(6, 12, n),
    'peer_feedback_score': np.random.uniform(1, 5, n),
    'manager_score': np.random.uniform(1, 5, n),
    'kudos_count': np.random.randint(0, 20, n),
    'promotions_in_2y': np.random.randint(0, 3, n),
    'salary_percentile_band': np.random.choice(['Low', 'Medium', 'High'], n)
}

df = pd.DataFrame(data)

# Simulate performance band
def calculate_performance(row):
    score = (row['on_time_delivery_rate'] * 20 + row['code_review_score'] * 10 + 
             row['peer_feedback_score'] * 10 + row['manager_score'] * 15 + 
             row['training_hours'] / 10 + row['certifications_count'] * 5 - 
             row['avg_task_delay_days'] / 2 - row['bug_count'] / 5 - 
             row['sick_days'] - row['unplanned_absences'] * 2)
    if row['experience_years'] > 10: score += 10
    if row['job_level'] == 'Senior': score += 5
    score += np.random.normal(0, 10)
    return 'High' if score > 70 else 'Medium' if score > 40 else 'Low'

df['perf_band_next'] = df.apply(calculate_performance, axis=1)
df.to_csv('../data/employee_features.csv', index=False)
print("Dataset created and saved.")

In [ ]:
# Import Libraries and Set Notebook Environment
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Display settings
pd.set_option('display.max_columns', None)
print("Libraries imported successfully.")

# Employee Performance Predictor - EDA Notebook

This notebook performs exploratory data analysis on the synthetic employee dataset to understand distributions, correlations, and feature importance for predicting performance bands.